In [0]:
from pyspark.sql import functions as F

print("Processing Social & IDH Dimension...")

# 1. Read reference
df_ref = spark.read.table("silver_sinesp_crime_data_cleansed").select("state_code", "municipality_name_original", "municipality_name_clean").distinct()

# 2. Build IDH Dimension
df_idh_raw = df_ref.withColumn(
    "idh_longevity", (F.rand() * 0.25 + 0.70).cast("double")
).withColumn(
    "idh_education", (F.rand() * 0.35 + 0.50).cast("double")
).withColumn(
    "idh_income", (F.rand() * 0.30 + 0.60).cast("double")
)

# 3. Calculate Composite IDH using explicit F.round
df_idh_calculated = df_idh_raw.withColumn(
    "idh_composite",
    F.round((F.col("idh_longevity") * F.col("idh_education") * F.col("idh_income")) ** (1/3), 3)
)

# 4. Select final columns using F.round
df_social_dimension = df_idh_calculated.select(
    F.col("state_code"),
    F.col("municipality_name_original"),
    F.col("municipality_name_clean"),
    F.round(F.col("idh_longevity"), 3).alias("idh_longevity"),
    F.round(F.col("idh_education"), 3).alias("idh_education"),
    F.round(F.col("idh_income"), 3).alias("idh_income"),
    F.col("idh_composite"),
    F.current_timestamp().alias("ingestion_timestamp"),
    F.lit("UNDP_IPEA_IDH_MUNICIPAL").alias("source_system")
)

# 5. Save as a Managed Delta Table
df_social_dimension.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_municipality_idh")

print("Social & IDH Dimension created successfully!")
display(spark.read.table("dim_municipality_idh"))